## Example 1: libarchive_c functionality test 1

In [ ]:
import libarchive
import tempfile
import os

# Example 1: Basic archive reading and file iteration
print("Example 1: Basic Archive Reading")

# Create a simple test file for demonstration
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write("Hello, this is test content for libarchive!")
    test_file = f.name

# Create a tar archive containing the test file
archive_path = tempfile.mktemp(suffix='.tar')
with libarchive.file_writer(archive_path, 'ustar') as archive:
    archive.add_files(test_file)

print(f"Created archive: {os.path.basename(archive_path)}")

# Read the archive and iterate through entries
with libarchive.file_reader(archive_path) as archive:
    for entry in archive:
        print(f"  Entry: {entry.name}")
        print(f"  Size: {entry.size} bytes")
        print(f"  Type: {'File' if entry.isfile else 'Directory'}")
        
        # Read the content
        content = b''.join(entry.get_blocks())
        print(f"  Content: {content.decode()[:50]}...")

# Cleanup
os.unlink(test_file)
os.unlink(archive_path)
print("Archive reading completed successfully")

## Example 2: libarchive_c functionality test 2

In [ ]:
import libarchive
import tempfile
import os

# Example 2: Archive creation with multiple formats
print("Example 2: Archive Creation with Multiple Formats")

# Create test files
test_files = []
for i in range(3):
    with tempfile.NamedTemporaryFile(mode='w', suffix=f'_test{i}.txt', delete=False) as f:
        f.write(f"Test file {i} content\nLine 2 of file {i}\n")
        test_files.append(f.name)

print(f"Created {len(test_files)} test files")

# Create different archive formats
formats = [
    ('gzipped_tar', 'tar.gz', 'ustar', 'gzip'),
    ('bzip2_tar', 'tar.bz2', 'ustar', 'bzip2'),
    ('plain_tar', 'tar', 'ustar', None)
]

for name, ext, format_name, compression in formats:
    archive_path = tempfile.mktemp(suffix=f'.{ext}')
    
    # Create archive with compression if specified
    if compression:
        with libarchive.file_writer(archive_path, format_name, compression) as archive:
            archive.add_files(*test_files)
    else:
        with libarchive.file_writer(archive_path, format_name) as archive:
            archive.add_files(*test_files)
    
    # Check the created archive
    size = os.path.getsize(archive_path)
    print(f"  {name}: {size} bytes ({ext})")
    
    os.unlink(archive_path)

# Cleanup test files
for f in test_files:
    os.unlink(f)

print("Archive creation with different formats completed")

## Example 3: libarchive_c functionality test 3

In [ ]:
import libarchive
import tempfile
import os

# Example 3: Memory-based archive operations
print("Example 3: Memory-based Archive Operations")

# Create test data in memory
test_data = {
    'file1.txt': b'Content of file 1\nSecond line\n',
    'file2.txt': b'Different content in file 2\n',
    'data.bin': bytes(range(256))  # Binary data
}

print(f"Created {len(test_data)} files in memory")

# Create archive in memory
archive_data = b''
with tempfile.NamedTemporaryFile() as temp_archive:
    # Use file_writer to create archive, then read it back
    with libarchive.file_writer(temp_archive.name, 'ustar') as archive:
        for filename, content in test_data.items():
            # Write content to temporary file first
            with tempfile.NamedTemporaryFile(delete=False) as temp_file:
                temp_file.write(content)
                temp_file.flush()
                archive.add_files(temp_file.name)
                os.unlink(temp_file.name)
    
    # Read archive data
    with open(temp_archive.name, 'rb') as f:
        archive_data = f.read()

print(f"Archive created in memory: {len(archive_data)} bytes")

# Read archive from memory
with libarchive.memory_reader(archive_data) as archive:
    print("Reading archive from memory:")
    for entry in archive:
        print(f"  Entry: {entry.name}")
        print(f"  Size: {entry.size} bytes")
        
        # Read content from archive
        content = b''.join(entry.get_blocks())
        if entry.name.endswith('.txt'):
            print(f"  Content preview: {content.decode()[:30]}...")
        else:
            print(f"  Binary content: {len(content)} bytes")

print("Memory-based archive operations completed")

## Example 4: libarchive_c functionality test 4

In [ ]:
import libarchive
import tempfile
import os
import shutil

# Example 4: Archive extraction to directory
print("Example 4: Archive Extraction to Directory")

# Create a source directory with files
source_dir = tempfile.mkdtemp(prefix='libarchive_source_')
for i in range(3):
    subdir = os.path.join(source_dir, f'subdir_{i}')
    os.makedirs(subdir)
    
    # Create files in subdirectory
    for j in range(2):
        file_path = os.path.join(subdir, f'file_{j}.txt')
        with open(file_path, 'w') as f:
            f.write(f'Content of file {j} in subdirectory {i}\n')

print(f"Created source directory with nested structure")

# Create archive from directory
archive_path = tempfile.mktemp(suffix='.tar.gz')
with libarchive.file_writer(archive_path, 'ustar', 'gzip') as archive:
    archive.add_files(source_dir)

print(f"Created compressed archive: {os.path.getsize(archive_path)} bytes")

# Extract to a new directory using extract_file
extract_dir = tempfile.mkdtemp(prefix='libarchive_extract_')
old_cwd = os.getcwd()
try:
    os.chdir(extract_dir)
    libarchive.extract_file(archive_path)
    
    # List extracted contents
    print(f"Extracted to: {extract_dir}")
    for root, dirs, files in os.walk(extract_dir):
        level = root.replace(extract_dir, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 2 * (level + 1)
        for file in files:
            print(f"{subindent}{file}")

finally:
    os.chdir(old_cwd)

# Cleanup
shutil.rmtree(source_dir)
shutil.rmtree(extract_dir)
os.unlink(archive_path)
print("Archive extraction completed successfully")

## Example 5: libarchive_c functionality test 5

In [ ]:
import libarchive
import tempfile
import os
import time

# Example 5: Archive metadata and entry information
print("Example 5: Archive Metadata and Entry Information")

# Create test files with different attributes
test_files = []
file_contents = [
    "Small text file content",
    "A" * 1000,  # Larger content
    "#!/bin/bash\necho 'executable script'\n"  # Script-like content
]

for i, content in enumerate(file_contents):
    with tempfile.NamedTemporaryFile(mode='w', suffix=f'_test{i}.txt', delete=False) as f:
        f.write(content)
        test_files.append(f.name)

# Make one file executable
os.chmod(test_files[2], 0o755)

print(f"Created {len(test_files)} test files")

# Create archive
archive_path = tempfile.mktemp(suffix='.tar')
with libarchive.file_writer(archive_path, 'ustar') as archive:
    archive.add_files(*test_files)

# Read archive and examine metadata
print(f"\nArchive metadata analysis:")
total_size = 0
with libarchive.file_reader(archive_path) as archive:
    for i, entry in enumerate(archive):
        print(f"\nEntry {i+1}: {os.path.basename(entry.name)}")
        print(f"  Size: {entry.size} bytes")
        print(f"  Type: {'File' if entry.isfile else 'Directory' if entry.isdir else 'Other'}")
        print(f"  Mode: {oct(entry.mode) if hasattr(entry, 'mode') else 'N/A'}")
        print(f"  UID/GID: {getattr(entry, 'uid', 'N/A')}/{getattr(entry, 'gid', 'N/A')}")
        
        # Read and analyze content
        content = b''.join(entry.get_blocks())
        total_size += len(content)
        
        if entry.name.endswith('.txt'):
            lines = content.decode().split('\n')
            print(f"  Lines: {len(lines)}")
            print(f"  Preview: {content.decode()[:40]}...")

print(f"\nArchive summary:")
print(f"  Total entries: {i+1}")
print(f"  Total content size: {total_size} bytes")
print(f"  Archive file size: {os.path.getsize(archive_path)} bytes")

# Cleanup
for f in test_files:
    os.unlink(f)
os.unlink(archive_path)
print("\nMetadata analysis completed")